In [ ]:
print("STARTUP: CPU mount validation; no training or GPU allocation", flush=True)
from pathlib import Path
import hashlib, json
import numpy as np
root=Path('/kaggle/input')
def locate(slug, owner):
    paths=[root/'datasets'/owner/slug,root/slug]
    found=[p for p in paths if p.is_dir()]
    if len(found)!=1: raise FileNotFoundError(f'{slug}: {found}')
    return found[0]
source=locate('rsna-cnn-hpo-blend-xla-source-v1','tumis001')/'skanee'
trainer=source/'scripts/train_cnn_baseline.py'
assert hashlib.sha256(trainer.read_bytes()).hexdigest()=='b8f8302319afbcf9970f0cf5b806d8fb78e72ef6e169b276274ddf7cd5e0ba85'
compile(trainer.read_text(),str(trainer),'exec')
labels=locate('rsna-knee-hpo-v31-d2-report-labels','tumis001')/'soft_labels_hpo_v3_1_d2_structured.json'
assert hashlib.sha256(labels.read_bytes()).hexdigest()=='38367d62eb9e7c198b69d88e2b9ad419e00fea9d1211b98feae45cd8a386f92d'
assert (locate('rsna-knee-llm-report-labels','stevenleehans')/'llm_labels_v4_blend.csv').is_file()
for i in range(3):
    directory=locate(f'rsna-cnn-topk-support-cache-shard-{i}-rehost','tumis001')
    assert (directory/'manifest.jsonl').is_file() and (directory/'cache_summary.json').is_file()
    for kind in ['slices','support']:
        for part in range(4):
            array=np.load(directory/f'{kind}_{part:03d}.npy',mmap_mode='r',allow_pickle=False)
            assert array.size>0
            print(f'READABLE shard={i} {kind}_{part:03d} shape={array.shape}',flush=True)
    print(f'Shard{i} validated',flush=True)
Path('/kaggle/working/startup_verified.json').write_text(json.dumps({'startup_verified':True,'shards':3,'labels_sha256':hashlib.sha256(labels.read_bytes()).hexdigest()}))
print('STARTUP_VERIFIED: all required datasets mounted; source and labels match; no raw competition input needed',flush=True)
